In [1]:
import requests
import pandas as pd
import numpy as np

In [7]:
# =========================================================
# 1) FUNÇÕES DE COLETA
# =========================================================

def get_rain_data(hours=6):
    """
    Baixa dados da API de chuva.
    Retorna uma lista de medições.
    """
    url = "https://apps.spaguas.sp.gov.br/sibh/api/v2/measurements/now"
    params = {
        "station_type_id": 1,
        "hours": hours,
        "show_all": "true",
        "serializer": "complete",
        "public": "true"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    return data.get("measurements", [])


def get_level_data():
    """
    Baixa dados da API de nível.
    Retorna uma lista de medições.
    """
    url = "https://apps.spaguas.sp.gov.br/sibh/api/v2/measurements/now_flu"

    # Como o parâmetro references[] aparece repetido,
    # usamos lista de tuplas.
    params = [
        ("references[]", "extravasation"),
        ("references[]", "emergency"),
        ("references[]", "alert"),
        ("references[]", "attention"),
        ("with_one_ref", "true")
    ]

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    # A API pode retornar lista diretamente ou dentro de alguma chave.
    if isinstance(data, dict):
        for key in ["measurements", "data", "results"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    if isinstance(data, list):
        return data

    return []


In [8]:
# =========================================================
# 2) NORMALIZAÇÃO DOS DADOS
# =========================================================

def prepare_rain_df(rain_data):
    """
    Transforma JSON de chuva em DataFrame padronizado.
    """
    df = pd.json_normalize(rain_data)

    # Tentamos garantir nomes mínimos
    expected_columns = ["station_name", "city", "value"]
    for col in expected_columns:
        if col not in df.columns:
            df[col] = None

    # Converte chuva para número
    df["rain_value"] = pd.to_numeric(df["value"], errors="coerce")

    # Renomeia algumas colunas para evitar conflito com a API de nível
    df = df.rename(columns={
        "station_name": "station_name",
        "city": "city"
    })

    # Mantemos só algumas colunas relevantes
    keep_cols = [c for c in [
        "station_name", "city", "rain_value", "value", "max_value", "max_date", "min_value", "min_date"
    ] if c in df.columns]

    return df[keep_cols].copy()


def prepare_level_df(level_data):
    """
    Transforma JSON de nível em DataFrame padronizado.
    """
    df = pd.json_normalize(level_data)

    # Garantimos colunas mínimas
    expected_columns = ["station_name", "city", "value", "attention", "alert", "emergency", "extravasation"]
    for col in expected_columns:
        if col not in df.columns:
            df[col] = None

    # Nível atual
    df["level_value"] = pd.to_numeric(df["value"], errors="coerce")

    # Referências de nível
    for ref_col in ["attention", "alert", "emergency", "extravasation"]:
        df[ref_col] = pd.to_numeric(df[ref_col], errors="coerce")

    keep_cols = [c for c in [
        "station_name", "city", "level_value", "attention", "alert", "emergency", "extravasation"
    ] if c in df.columns]

    return df[keep_cols].copy()



In [9]:
# =========================================================
# 3) REGRA DE RISCO
# =========================================================

def calculate_level_ratio(row):
    """
    Calcula quão próximo o nível atual está das referências.
    Quanto maior esse valor, maior o risco potencial.

    Exemplo:
    - se nível atual >= attention -> já existe algum cuidado
    - se nível atual >= alert -> piora
    - se nível atual >= emergency -> mais grave
    - se nível atual >= extravasation -> crítico
    """
    level_value = row.get("level_value", np.nan)

    if pd.isna(level_value):
        return 0.0

    score = 0.0

    attention = row.get("attention", np.nan)
    alert = row.get("alert", np.nan)
    emergency = row.get("emergency", np.nan)
    extravasation = row.get("extravasation", np.nan)

    if not pd.isna(attention) and attention != 0:
        score = max(score, level_value / attention)

    if not pd.isna(alert) and alert != 0:
        score = max(score, level_value / alert)

    if not pd.isna(emergency) and emergency != 0:
        score = max(score, level_value / emergency)

    if not pd.isna(extravasation) and extravasation != 0:
        score = max(score, level_value / extravasation)

    return score


def classify_risk(score):
    """
    Classifica o risco final.
    Você pode ajustar esses thresholds depois.
    """
    if score >= 0.95:
        return "CRÍTICO"
    elif score >= 0.75:
        return "ALTO"
    elif score >= 0.45:
        return "MÉDIO"
    else:
        return "BAIXO"



In [10]:
# =========================================================
# 4) PIPELINE PRINCIPAL
# =========================================================

def build_risk_model(hours=6):
    """
    Executa o fluxo completo:
    - baixa chuva
    - baixa nível
    - junta dados
    - calcula score final
    """
    rain_data = get_rain_data(hours=hours)
    level_data = get_level_data()

    rain_df = prepare_rain_df(rain_data)
    level_df = prepare_level_df(level_data)

    # Junta por station_name e city
    # Dependendo da API real, você pode precisar ajustar a chave de merge.
    merged = pd.merge(
        rain_df,
        level_df,
        on=["station_name", "city"],
        how="outer"
    )

    # Preenche nulos com zero quando fizer sentido
    merged["rain_value"] = merged["rain_value"].fillna(0)
    merged["level_value"] = merged["level_value"].fillna(0)

    # Normalização simples da chuva
    # Isso ajuda a transformar chuva em um valor comparável
    max_rain = merged["rain_value"].max() if len(merged) > 0 else 0

    if max_rain and max_rain > 0:
        merged["rain_score"] = merged["rain_value"] / max_rain
    else:
        merged["rain_score"] = 0

    # Score do nível
    merged["level_score"] = merged.apply(calculate_level_ratio, axis=1)

    # Limitamos o score de nível para evitar explosões
    merged["level_score"] = merged["level_score"].clip(upper=1.5)

    # Score final ponderado
    # Aqui damos mais peso ao nível, mas você pode inverter.
    merged["risk_score"] = (
        merged["rain_score"] * 0.4 +
        merged["level_score"] * 0.6
    )

    # Classificação
    merged["risk_class"] = merged["risk_score"].apply(classify_risk)

    # Ordena do maior risco para o menor
    merged = merged.sort_values("risk_score", ascending=False)

    return merged


In [11]:
# =========================================================
# 5) EXECUÇÃO
# =========================================================

if __name__ == "__main__":
    df_risk = build_risk_model(hours=6)

    print("\n=== TOP 20 ESTAÇÕES POR RISCO ===\n")
    print(
        df_risk[
            [
                "station_name",
                "city",
                "rain_value",
                "level_value",
                "rain_score",
                "level_score",
                "risk_score",
                "risk_class"
            ]
        ].head(20).to_string(index=False)
    )

    # Salva resultado em CSV
    df_risk.to_csv("modelo_risco_chuva_nivel.csv", index=False, encoding="utf-8-sig")
    print("\nArquivo salvo: modelo_risco_chuva_nivel.csv")


=== TOP 20 ESTAÇÕES POR RISCO ===

                                            station_name                  city    rain_value  level_value  rain_score  level_score  risk_score risk_class
                     Campos do Jordão (Captação do Fojo)      Campos do Jordão 161108.055556     161107.9    1.000000     0.999683    0.999810    CRÍTICO
                         Rancharia (PCH Pari Barramento)             Rancharia  37217.500000      37218.0    0.231010     1.500000    0.992404    CRÍTICO
       São José dos Campos (UHE Jaguari Fazenda Santana)   São José dos Campos   1522.333333       1587.0    0.009449     1.500000    0.903780       ALTO
       Auriflama (UHE Ilha Solteira Fazenda Palmeirinha)             Auriflama    390.000000        381.0    0.002421     1.500000    0.900968       ALTO
      Apiaí (Captação ETA Corrego Ranchinho Agua Grande)                 Apiaí  97269.755556      97268.9    0.603755     0.999208    0.841027       ALTO
                          Piedade (Capta